In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

In [2]:
data_path = Path("../../data_treatement/qosbuddy_dataset_final.csv")
df = pd.read_csv(data_path)

print(df.shape)
df.head()

(54052, 19)


,rsrp,sinr,rsrq,throughput_dl,timestamp,lat,lon,source_file,label_call_drop_risk,label_low_throughput,label_handover_risk,label_coverage_hole,label_interference,label_voice_quality_risk,label_prb_congestion,any_degradation,severity,recommendation,throughput_ul
0,-93.69,-0.5,-12.31,4176.03,2021-08-15 16:49:38.500,6.437242,3.407646,mendeley.csv,False,False,False,False,True,False,False,1,WARNING,Nettoyage PCI + optimisation A3 offset + ajust...,NaN
1,-84.88,4.5,-8.06,NaN,2021-08-15 16:03:08.500,6.454035,3.389383,mendeley.csv,False,False,False,False,False,False,False,0,OK,Aucune dégradation détectée,NaN
2,-91.13,4.5,-10.06,5145.78,2021-08-16 16:04:33.500,6.456179,3.411872,mendeley.csv,False,False,False,False,False,False,False,0,OK,Aucune dégradation détectée,NaN
3,-116.00,-3.0,-17.00,13.00,2025-08-29 03:09:58.537,19.178959,83.410369,network_logs_1.csv,True,False,False,True,False,True,False,1,CRITICAL,Optimisation tilt + ajustement puissance cellu...,13.0
4,-107.00,-3.0,-15.00,15.00,2025-08-28 18:37:22.431,21.202232,81.761958,network_logs_1.csv,False,False,False,False,False,False,False,0,OK,Aucune dégradation détectée,3.0


In [3]:
print(df.columns.tolist())
print("\nData types:\n")
print(df.dtypes)

['rsrp', 'sinr', 'rsrq', 'throughput_dl', 'timestamp', 'lat', 'lon', 'source_file', 'label_call_drop_risk', 'label_low_throughput', 'label_handover_risk', 'label_coverage_hole', 'label_interference', 'label_voice_quality_risk', 'label_prb_congestion', 'any_degradation', 'severity', 'recommendation', 'throughput_ul']

Data types:

rsrp                        float64
sinr                        float64
rsrq                        float64
throughput_dl               float64
timestamp                    object
lat                         float64
lon                         float64
source_file                  object
label_call_drop_risk           bool
label_low_throughput           bool
label_handover_risk            bool
label_coverage_hole            bool
label_interference             bool
label_voice_quality_risk       bool
label_prb_congestion           bool
any_degradation               int64
severity                     object
recommendation               object
throughput_ul       

In [4]:
missing = df.isnull().sum().sort_values(ascending=False)
print(missing)

throughput_ul               40766
throughput_dl                1592
lat                           112
lon                           112
label_coverage_hole             0
recommendation                  0
severity                        0
any_degradation                 0
label_prb_congestion            0
label_voice_quality_risk        0
label_interference              0
rsrp                            0
label_handover_risk             0
sinr                            0
label_call_drop_risk            0
source_file                     0
timestamp                       0
rsrq                            0
label_low_throughput            0
dtype: int64


In [ ]:
#fill + missing flag for throughput_ul
df["throughput_ul_missing"] = df["throughput_ul"].isna().astype(int)
df["throughput_ul"] = df["throughput_ul"].fillna(df["throughput_ul"].median())

In [ ]:
#fill + missing flag for throughput_dl
df["throughput_dl_missing"] = df["throughput_dl"].isna().astype(int)
df["throughput_dl"] = df["throughput_dl"].fillna(df["throughput_dl"].median())

In [9]:
df = df.dropna(subset=["lat", "lon"]).copy()

In [10]:
missing = df.isnull().sum().sort_values(ascending=False)
print(missing)

rsrp                        0
label_voice_quality_risk    0
throughput_ul_missing       0
weekday                     0
month                       0
day                         0
hour                        0
throughput_ul               0
recommendation              0
severity                    0
any_degradation             0
label_prb_congestion        0
label_interference          0
sinr                        0
label_coverage_hole         0
label_handover_risk         0
label_low_throughput        0
label_call_drop_risk        0
source_file                 0
lon                         0
lat                         0
timestamp                   0
throughput_dl               0
rsrq                        0
throughput_dl_missing       0
dtype: int64


In [ ]:
#prepare timestamp
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

print(df["timestamp"].isnull().sum())
df = df.dropna(subset=["timestamp"]).copy()

df["hour"] = df["timestamp"].dt.hour
df["day"] = df["timestamp"].dt.day
df["month"] = df["timestamp"].dt.month
df["weekday"] = df["timestamp"].dt.weekday

0


In [11]:
target = "any_degradation"

feature_cols = [
    "rsrp",
    "sinr",
    "rsrq",
    "throughput_dl",
    "throughput_ul",
    "throughput_dl_missing",
    "throughput_ul_missing",
    "lat",
    "lon",
    "hour",
    "day",
    "month",
    "weekday"
]

df_model = df[feature_cols + [target]].copy()
df_model = df_model.dropna()

print(df_model.shape)
df_model.head()

(53940, 14)


,rsrp,sinr,rsrq,throughput_dl,throughput_ul,throughput_dl_missing,throughput_ul_missing,lat,lon,hour,day,month,weekday,any_degradation
0,-93.69,-0.5,-12.31,4176.03,13.0,0,1,6.437242,3.407646,16,15,8,6,1
1,-84.88,4.5,-8.06,7706.05,13.0,1,1,6.454035,3.389383,16,15,8,6,0
2,-91.13,4.5,-10.06,5145.78,13.0,0,1,6.456179,3.411872,16,16,8,0,0
3,-116.00,-3.0,-17.00,13.00,13.0,0,0,19.178959,83.410369,3,29,8,4,1
4,-107.00,-3.0,-15.00,15.00,3.0,0,0,21.202232,81.761958,18,28,8,3,0


In [12]:
#check class balance
print(df_model[target].value_counts(dropna=False))
print(df_model[target].value_counts(normalize=True, dropna=False))

any_degradation
0    37727
1    16213
Name: count, dtype: int64
any_degradation
0    0.699425
1    0.300575
Name: proportion, dtype: float64


In [13]:
X = df_model[feature_cols]
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

(43152, 13) (10788, 13)


In [14]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=200,
                       random_state=42)

In [15]:
y_pred = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 score:", f1_score(y_test, y_pred, average="weighted"))
print("\nClassification report:\n")
print(classification_report(y_test, y_pred))
print("\nConfusion matrix:\n")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9967556544308491
F1 score: 0.9967511896440089

Classification report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      7545
           1       1.00      0.99      0.99      3243

    accuracy                           1.00     10788
   macro avg       1.00      0.99      1.00     10788
weighted avg       1.00      1.00      1.00     10788


Confusion matrix:

[[7543    2]
 [  33 3210]]


In [16]:
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_model.feature_importances_
}).sort_values(by="importance", ascending=False)

importance_df

,feature,importance
1,sinr,0.477351
2,rsrq,0.212655
0,rsrp,0.112599
3,throughput_dl,0.070367
7,lat,0.040021
8,lon,0.030588
4,throughput_ul,0.023295
9,hour,0.013804
10,day,0.011917
6,throughput_ul_missing,0.004703
